In [ ]:
from src.data_utils import load_openrouter_models, load_whitelist, normalize_slug_advanced
from src.viz_utils import plot_clusters, interactive_scatter
import pandas as pd


In [ ]:
url = "https://openrouter.ai/api/v1/models"

In [ ]:
request = requests.get(url)
openrouter_data = request.json()
model_list = openrouter_data['data']

In [ ]:
print(len(model_list))

In [ ]:
print(json.dumps(model_list[0], indent=2))

In [ ]:
url_whitelist = "https://cdn.reply.com/documents/challenges/02_26/api_model_whitelist.html"

In [ ]:
request_wl = requests.get(url_whitelist)
soup = BeautifulSoup(request_wl.text, 'html.parser')

model_name = soup.get_text(separator='\n')

whitelist_models = []
for line in model_name.split('\n'):
    line = line.strip()
    if '/' in line and ' ' not in line and not line.startswith('http'):
        whitelist_models.append(line)

In [ ]:
print(whitelist_models[0])

In [ ]:
def normalize_slug_advanced(slug, is_aa=False):
    if not isinstance(slug, str) or slug == 'N/D':
        return slug

    s = slug.lower().replace('.', '-')

    if 'claude-haiku-4-5' in s:
        s = s.replace('claude-haiku-4-5', 'claude-4-5-haiku')

    if '-instruct' in s:
        s = s.replace('-instruct', '-it')

    if 'gemma' in s and s.endswith('-it'):
        s = s[:-3]

    suffixes_to_remove = ['-v1', '-preview']
    for suffix in suffixes_to_remove:
        if s.endswith(suffix):
            s = s.replace(suffix, '')

    s = s.replace('-thinking', '-reasoning')

    if is_aa:
        if s.endswith('-reasoning'):
            s = s.replace('-reasoning', '')

    parti = s.split('-')
    parti = [p for p in parti if p]

    parti.sort()
    s = '-'.join(parti)

    return s

In [ ]:
# normalize_slug_advanced moved to src/data_utils.py

In [ ]:
print(len(whitelist_data))

In [ ]:
df_or = pd.DataFrame(whitelist_data)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [ ]:
df_or = df_or.sort_values(by='Slug_Join')

df_or = df_or.reset_index(drop=True)

display(df_or)

In [ ]:
url_aa = "https://artificialanalysis.ai/api/v2/data/llms/models"
headers_aa = {
    "x-api-key": "aa_fxsMhDDowmnvdaBNtLieitdtcZWpENvO"
}

response_aa = requests.get(url_aa, headers=headers_aa)

In [ ]:
if response_aa.status_code == 200:
    data_aa = response_aa.json().get('data', [])

    metrics_aa = []
    for mod in data_aa:
      evals = mod.get('evaluations', {})
      raw_slug = mod.get('slug', '')
      slug_final = normalize_slug_advanced(raw_slug, is_aa=True)

      metrics_aa.append({
            'Model Name (AA)': mod.get('name', 'N/D'),
            'Slug_Join': slug_final,
            'Intelligence Index': evals.get('artificial_analysis_intelligence_index', 'N/D'),
            'Coding Index' : evals.get('artificial_analysis_coding_index', 'N/D'),
            'Math Index (Reasoning)': evals.get('artificial_analysis_math_index', 'N/D'),
            'MMLU Pro (Knowledge)': evals.get('mmlu_pro', 'N/D'),
            'GPQA (Knowledge)': evals.get('gpqa', 'N/D'),
            'Speed (Tokens/s)': mod.get('median_output_tokens_per_second', 'N/D'),
            'Latency (s)': mod.get('median_time_to_first_token_seconds', 'N/D')})
else:
    print(f"❌ Errore API AA: {response_aa.status_code} - {response_aa.text}")

In [ ]:
df_aa = pd.DataFrame(metrics_aa)

In [ ]:
df_aa = df_aa.sort_values(by='Slug_Join')

df_aa = df_aa.reset_index(drop=True)

display(df_aa)

In [ ]:
#JOIN

df = pd.merge(df_or, df_aa, on='Slug_Join', how='left')

df = df.drop(columns=['Slug_Join'])

In [ ]:
df = df.sort_values(by='Link')

df = df.reset_index(drop=True)

display(df)

In [ ]:
vuoti_per_colonna = df.isna().sum()

print("Celle veramente vuote (NaN) per colonna:")
print(vuoti_per_colonna)

In [ ]:
nome_file_finale = "models.xlsx"

# Salva il DataFrame in CSV
df.to_excel(nome_file_finale, index=False)

DATA ANALYSIS

In [ ]:
import numpy as np

In [ ]:
df = df.replace('N/D', np.nan)

colonne_da_dividere = [
    'Intelligence Index',
    'Coding Index',
    'Math Index (Reasoning)'
]

for col in colonne_da_dividere:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

        df[col] = df[col] / 100

In [ ]:
display(df[['Link', 'Intelligence Index', 'Coding Index', 'Math Index (Reasoning)']].head(10))

In [ ]:
import numpy as np

colonne_tempo = ['Speed (Tokens/s)', 'Latency (s)']

for col in colonne_tempo:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].replace(0, np.nan)

def min_max_norm(serie):
    s = pd.to_numeric(serie, errors='coerce')
    return (s - s.min()) / (s.max() - s.min())


if 'Context Window (Token)' in df.columns:
    df['Norm_Context'] = min_max_norm(df['Context Window (Token)'])

if 'Speed (Tokens/s)' in df.columns:
    df['Norm_Speed'] = min_max_norm(df['Speed (Tokens/s)'])

if 'Latency (s)' in df.columns:
    df['Norm_Latency'] = 1 - min_max_norm(df['Latency (s)'])

if 'Costo Input (1M Token) $' in df.columns:
    df['Norm_Cost_In'] = 1 - min_max_norm(df['Costo Input (1M Token) $'])

if 'Costo Output (1M Token) $' in df.columns:
    df['Norm_Cost_Out'] = 1 - min_max_norm(df['Costo Output (1M Token) $'])

colonne_norm = [col for col in df.columns if col.startswith('Norm_')]
display(df[['Link'] + colonne_norm].head(10))

In [ ]:
# ==========================================
# 1. PARSER / ROUTER
# ==========================================
df['Score_Parser'] = (
    df['Norm_Cost_In'] * 0.30 +
    df['Norm_Cost_Out'] * 0.15 +
    df['Norm_Latency'] * 0.25 +
    df['Norm_Speed'] * 0.15 +
    df.get('Intelligence Index', 0) * 0.15
)

# ==========================================
# 2. WORKER (DATA MANIPULATOR)
# ==========================================
# Usiamo il nome colonna corretto 'Coding Index' e gestiamo il caso in cui Math Index (Reasoning) non esista
logica_worker = df['Coding Index']
if 'Math Index (Reasoning)' in df.columns:
    logica_worker = df[['Coding Index', 'Math Index (Reasoning)']].mean(axis=1)

df['Score_Worker'] = (
    logica_worker * 0.35 +
    df['Norm_Cost_In'] * 0.20 +
    df['Norm_Cost_Out'] * 0.10 +
    df['Norm_Speed'] * 0.15 +
    df['Norm_Latency'] * 0.10 +
    df['Norm_Context'] * 0.05 +
    df.get('Intelligence Index', 0) * 0.05
)

# ==========================================
# 3. ORCHESTRATOR
# ==========================================
df['Score_Orchestrator'] = (
    df.get('Intelligence Index', 0) * 0.55 +
    df.get('Math Index (Reasoning)', 0) * 0.05 +
    df.get('MMLU Pro (Knowledge)', 0) * 0.05 +
    df['Norm_Context'] * 0.10 +
    df['Norm_Cost_In'] * 0.05 +
    df['Norm_Cost_Out'] * 0.05 +
    df['Norm_Speed'] * 0.15
)

# ==========================================
# STAMPA DEI RISULTATI
# ==========================================
def stampa_top(ruolo, colonna_score, colonne_extra):
    df_valido = df.dropna(subset=[colonna_score]).copy()
    top = df_valido.sort_values(by=colonna_score, ascending=False).head(5)

    top[colonna_score] = (top[colonna_score] * 100).round(1)

    # Gestione nomi colonne per la stampa
    colonne_presenti = [c for c in colonne_extra if c in top.columns]
    colonna_nome = 'Model Name (AA)' if 'Model Name (AA)' in top.columns else 'Link'

    display(top[[colonna_nome, colonna_score] + colonne_presenti])

print("\n✰ TOP 5 - AGENTI PARSER / ROUTER")
stampa_top('Parser', 'Score_Parser', ['Costo Input (1M Token) $', 'Latency (s)', 'Speed (Tokens/s)'])

print("\n⚙‸ TOP 5 - AGENTI WORKER")
stampa_top('Worker', 'Score_Worker', ['Coding Index', 'Costo Input (1M Token) $', 'Latency (s)'])

print("\n   TOP 5 - AGENTI ORCHESTRATOR")
stampa_top('Orchestrator', 'Score_Orchestrator', ['Intelligence Index', 'Context Window (Token)', 'Costo Input (1M Token) $'])


In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

feature_cols = [
    'Intelligence Index', 'Coding Index', 'Math Index (Reasoning)', 'MMLU Pro (Knowledge)',
    'Norm_Speed', 'Norm_Context', 'Norm_Cost_In', 'Norm_Cost_Out', 'Norm_Latency'
]

df_clean = df.dropna(subset=feature_cols).copy()

X = df_clean[feature_cols]

K = 9

kmeans = KMeans(n_clusters=K, random_state=42, n_init='auto')
df_clean['Cluster_ID'] = kmeans.fit_predict(X)

df['Cluster_ID'] = np.nan
df.update(df_clean[['Cluster_ID']])

cluster_profiles = df_clean.groupby('Cluster_ID')[feature_cols].mean()

new_names = [
    'Intelligence Index', 'Coding Index', 'Math Index (Reasoning)', 'MMLU Pro (Knowledge)',
    'Speed (Tokens/s)', 'Context', 'Cost In (Token)', 'Cost Out (Token)', 'Latency'
]
cluster_profiles.columns = new_names

plt.figure(figsize=(12, 6))
sns.heatmap(cluster_profiles, annot=True, cmap='coolwarm', fmt=".2f", vmin=0, vmax=1)
plt.title(f"Cluster (K={K})")
plt.ylabel("ID Cluster")
plt.tight_layout()
plt.show()

for i in range(K):
    print(f"\n🏷️ ESEMPI DI MODELLI NEL CLUSTER {i}:")
    modelli_cluster = df_clean[df_clean['Cluster_ID'] == i]
    colonna_nome = 'Link' if 'Link' in modelli_cluster.columns else modelli_cluster.columns[0]
    display(modelli_cluster[[colonna_nome, 'Costo Input (1M Token) $', 'Intelligence Index']].head(10))

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

metriche = [
    'Costo Input (1M Token) $', 'Costo Output (1M Token) $', 'Latency (s)',
    'Speed (Tokens/s)', 'Intelligence Index', 'Coding Index',
    'Math Index (Reasoning)', 'MMLU Pro (Knowledge)', 'Context Window (Token)'
]

da_minimizzare = ['Costo Input (1M Token) $', 'Costo Output (1M Token) $', 'Latency (s)']

metriche_log = [
    'Costo Input (1M Token) $', 'Costo Output (1M Token) $',
    'Latency (s)', 'Speed (Tokens/s)', 'Context Window (Token)'
]

out_grafico = widgets.Output()

def aggiorna_grafico(asse_x, asse_y):
    with out_grafico:
        clear_output(wait=True)

        if asse_x == asse_y:
            print("Scegli due metriche diverse per gli assi X e Y.")
            return

        df_plot = df.copy()
        df_plot[asse_x] = pd.to_numeric(df_plot[asse_x], errors='coerce')
        df_plot[asse_y] = pd.to_numeric(df_plot[asse_y], errors='coerce')
        df_plot = df_plot.dropna(subset=[asse_x, asse_y])

        if df_plot.empty:
            print("Nessun dato disponibile per questa combinazione.")
            return

        min_x = asse_x in da_minimizzare
        min_y = asse_y in da_minimizzare

        df_plot = df_plot.sort_values(by=[asse_x, asse_y], ascending=[min_x, min_y])

        pareto_front = []
        best_y = float('inf') if min_y else float('-inf')

        for index, row in df_plot.iterrows():
            val_y = row[asse_y]
            is_better = (val_y < best_y) if min_y else (val_y > best_y)
            if is_better:
                pareto_front.append(row)
                best_y = val_y

        df_frontiera = pd.DataFrame(pareto_front)

        colonna_nome = 'Model Name (AA)' if 'Model Name (AA)' in df_plot.columns else 'Link'
        fig = px.scatter(
            df_plot,
            x=asse_x,
            y=asse_y,
            hover_name=colonna_nome,
            title=f"{asse_x} vs {asse_y}",
            template='plotly_white',
            height=600,
            opacity=0.6
        )

        if asse_x in metriche_log:
            fig.update_layout(xaxis_type="log")
        if asse_y in metriche_log:
            fig.update_layout(yaxis_type="log")

        if not df_frontiera.empty:
            df_frontiera = df_frontiera.sort_values(by=asse_x)
            fig.add_trace(
                go.Scatter(
                    x=df_frontiera[asse_x],
                    y=df_frontiera[asse_y],
                    mode='lines+markers',
                    line=dict(color='red', width=2, shape='hv'),
                    name='Optimal models',
                    marker=dict(size=8, color='red'),
                    hoverinfo='skip'
                )
            )

        fig_widget = go.FigureWidget(fig)
        display(fig_widget)

        print(f"{len(df_frontiera)} MODELLI CON MIGLIORE TRADE_OFF:")
        display(df_frontiera[[colonna_nome, asse_x, asse_y]].reset_index(drop=True))

dropdown_x = widgets.Dropdown(options=metriche, value='Costo Input (1M Token) $', description='Asse X:')
dropdown_y = widgets.Dropdown(options=metriche, value='Intelligence Index', description='Asse Y:')

def on_menu_change(change):
    aggiorna_grafico(dropdown_x.value, dropdown_y.value)

dropdown_x.observe(on_menu_change, names='value')
dropdown_y.observe(on_menu_change, names='value')

menu_box = widgets.HBox([dropdown_x, dropdown_y])
display(menu_box)
display(out_grafico)

aggiorna_grafico(dropdown_x.value, dropdown_y.value)